# Inception V3 — From Scratch in PyTorch
**Paper:** Rethinking the Inception Architecture for Computer Vision (Szegedy et al., CVPR 2016)

| Config | Value |
|--------|-------|
| Inception Types | A (5×5 factorized), B (grid reduction), C (7×7 factorized) |
| Auxiliary Classifier | 1 |
| Input Size | 299×299 |
| Parameters | ~27M |

In [ ]:
# Cell 1 — Imports
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc, roc_auc_score
from sklearn.preprocessing import label_binarize

In [ ]:
# Cell 2 — Inception V3 Architecture (raw, no torchvision)
class ConvBnRelu(nn.Module):
    def __init__(self, in_channels, out_channels, **kwargs):
        super(ConvBnRelu, self).__init__()

        self.conv = nn.Conv2d(in_channels, out_channels, bias=False, **kwargs)
        self.bn   = nn.BatchNorm2d(out_channels)

    def forward(self, x):
        return F.relu(self.bn(self.conv(x)), inplace=True)


class InceptionA(nn.Module):
    """Factorizes 5x5 → two 3x3 convolutions."""
    def __init__(self, in_channels, pool_proj):
        super(InceptionA, self).__init__()

        self.branch1 = ConvBnRelu(in_channels, 64, kernel_size=1)
        self.branch2 = nn.Sequential(
            ConvBnRelu(in_channels, 48, kernel_size=1),
            ConvBnRelu(48, 64, kernel_size=5, padding=2),
        )
        self.branch3 = nn.Sequential(
            ConvBnRelu(in_channels, 64, kernel_size=1),
            ConvBnRelu(64, 96, kernel_size=3, padding=1),
            ConvBnRelu(96, 96, kernel_size=3, padding=1),
        )
        self.branch4 = nn.Sequential(
            nn.AvgPool2d(kernel_size=3, stride=1, padding=1),
            ConvBnRelu(in_channels, pool_proj, kernel_size=1),
        )

    def forward(self, x):
        return torch.cat([self.branch1(x), self.branch2(x),
                          self.branch3(x), self.branch4(x)], dim=1)


class InceptionB(nn.Module):
    """Grid size reduction block."""
    def __init__(self, in_channels):
        super(InceptionB, self).__init__()

        self.branch1 = ConvBnRelu(in_channels, 384, kernel_size=3, stride=2)
        self.branch2 = nn.Sequential(
            ConvBnRelu(in_channels, 64, kernel_size=1),
            ConvBnRelu(64, 96, kernel_size=3, padding=1),
            ConvBnRelu(96, 96, kernel_size=3, stride=2),
        )
        self.branch3 = nn.MaxPool2d(kernel_size=3, stride=2)

    def forward(self, x):
        return torch.cat([self.branch1(x), self.branch2(x), self.branch3(x)], dim=1)


class InceptionC(nn.Module):
    """Factorizes n×n → 1×n + n×1 convolutions."""
    def __init__(self, in_channels, channels_7x7):
        super(InceptionC, self).__init__()
        c7 = channels_7x7

        self.branch1 = ConvBnRelu(in_channels, 192, kernel_size=1)
        self.branch2 = nn.Sequential(
            ConvBnRelu(in_channels, c7, kernel_size=1),
            ConvBnRelu(c7, c7,  kernel_size=(1, 7), padding=(0, 3)),
            ConvBnRelu(c7, 192, kernel_size=(7, 1), padding=(3, 0)),
        )
        self.branch3 = nn.Sequential(
            ConvBnRelu(in_channels, c7, kernel_size=1),
            ConvBnRelu(c7, c7,  kernel_size=(7, 1), padding=(3, 0)),
            ConvBnRelu(c7, c7,  kernel_size=(1, 7), padding=(0, 3)),
            ConvBnRelu(c7, c7,  kernel_size=(7, 1), padding=(3, 0)),
            ConvBnRelu(c7, 192, kernel_size=(1, 7), padding=(0, 3)),
        )
        self.branch4 = nn.Sequential(
            nn.AvgPool2d(kernel_size=3, stride=1, padding=1),
            ConvBnRelu(in_channels, 192, kernel_size=1),
        )

    def forward(self, x):
        return torch.cat([self.branch1(x), self.branch2(x),
                          self.branch3(x), self.branch4(x)], dim=1)


class InceptionD(nn.Module):
    """Second grid size reduction block."""
    def __init__(self, in_channels):
        super(InceptionD, self).__init__()

        self.branch1 = nn.Sequential(
            ConvBnRelu(in_channels, 192, kernel_size=1),
            ConvBnRelu(192, 320, kernel_size=3, stride=2),
        )
        self.branch2 = nn.Sequential(
            ConvBnRelu(in_channels, 192, kernel_size=1),
            ConvBnRelu(192, 192, kernel_size=(1, 7), padding=(0, 3)),
            ConvBnRelu(192, 192, kernel_size=(7, 1), padding=(3, 0)),
            ConvBnRelu(192, 192, kernel_size=3, stride=2),
        )
        self.branch3 = nn.MaxPool2d(kernel_size=3, stride=2)

    def forward(self, x):
        return torch.cat([self.branch1(x), self.branch2(x), self.branch3(x)], dim=1)


class InceptionE(nn.Module):
    """Wide inception with parallel 1×3 and 3×1 branches."""
    def __init__(self, in_channels):
        super(InceptionE, self).__init__()

        self.branch1 = ConvBnRelu(in_channels, 320, kernel_size=1)
        self.branch2_conv = ConvBnRelu(in_channels, 384, kernel_size=1)
        self.branch2a     = ConvBnRelu(384, 384, kernel_size=(1, 3), padding=(0, 1))
        self.branch2b     = ConvBnRelu(384, 384, kernel_size=(3, 1), padding=(1, 0))
        self.branch3_conv = nn.Sequential(
            ConvBnRelu(in_channels, 448, kernel_size=1),
            ConvBnRelu(448, 384, kernel_size=3, padding=1),
        )
        self.branch3a     = ConvBnRelu(384, 384, kernel_size=(1, 3), padding=(0, 1))
        self.branch3b     = ConvBnRelu(384, 384, kernel_size=(3, 1), padding=(1, 0))
        self.branch4      = nn.Sequential(
            nn.AvgPool2d(kernel_size=3, stride=1, padding=1),
            ConvBnRelu(in_channels, 192, kernel_size=1),
        )

    def forward(self, x):
        b1  = self.branch1(x)
        b2  = self.branch2_conv(x)
        b2  = torch.cat([self.branch2a(b2), self.branch2b(b2)], dim=1)
        b3  = self.branch3_conv(x)
        b3  = torch.cat([self.branch3a(b3), self.branch3b(b3)], dim=1)
        b4  = self.branch4(x)
        return torch.cat([b1, b2, b3, b4], dim=1)


class AuxClassifier(nn.Module):
    def __init__(self, in_channels, num_classes):
        super(AuxClassifier, self).__init__()

        self.pool    = nn.AdaptiveAvgPool2d((5, 5))
        self.conv    = ConvBnRelu(in_channels, 128, kernel_size=1)
        self.fc1     = nn.Linear(128 * 5 * 5, 1024)
        self.dropout = nn.Dropout(p=0.7)
        self.fc2     = nn.Linear(1024, num_classes)

    def forward(self, x):
        x = self.pool(x)
        x = self.conv(x)
        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x), inplace=True)
        x = self.dropout(x)
        x = self.fc2(x)
        return x


class InceptionV3(nn.Module):
    def __init__(self, num_classes=1000, use_aux=True):
        super(InceptionV3, self).__init__()
        self.use_aux = use_aux

        self.stem = nn.Sequential(
            ConvBnRelu(3, 32, kernel_size=3, stride=2),
            ConvBnRelu(32, 32, kernel_size=3),
            ConvBnRelu(32, 64, kernel_size=3, padding=1),
            nn.MaxPool2d(kernel_size=3, stride=2),
            ConvBnRelu(64, 80, kernel_size=1),
            ConvBnRelu(80, 192, kernel_size=3),
            nn.MaxPool2d(kernel_size=3, stride=2),
        )

        self.inceptionA1 = InceptionA(192, pool_proj=32)
        self.inceptionA2 = InceptionA(256, pool_proj=64)
        self.inceptionA3 = InceptionA(288, pool_proj=64)
        self.inceptionB  = InceptionB(288)

        self.inceptionC1 = InceptionC(768, channels_7x7=128)
        self.inceptionC2 = InceptionC(768, channels_7x7=160)
        self.inceptionC3 = InceptionC(768, channels_7x7=160)
        self.inceptionC4 = InceptionC(768, channels_7x7=192)

        if self.use_aux:
            self.aux = AuxClassifier(768, num_classes)

        self.inceptionD  = InceptionD(768)
        self.inceptionE1 = InceptionE(1280)
        self.inceptionE2 = InceptionE(2048)

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.dropout = nn.Dropout(p=0.5)
        self.fc      = nn.Linear(2048, num_classes)

    def forward(self, x):
        x = self.stem(x)

        x = self.inceptionA1(x)
        x = self.inceptionA2(x)
        x = self.inceptionA3(x)
        x = self.inceptionB(x)

        x = self.inceptionC1(x)
        x = self.inceptionC2(x)
        x = self.inceptionC3(x)
        x = self.inceptionC4(x)
        aux_out = self.aux(x) if (self.use_aux and self.training) else None

        x = self.inceptionD(x)
        x = self.inceptionE1(x)
        x = self.inceptionE2(x)

        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.dropout(x)
        x = self.fc(x)

        if self.use_aux and self.training:
            return x, aux_out
        return x

In [ ]:
# Cell 3 — Instantiate Model
NUM_CLASSES = 10
DEVICE      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')

model = InceptionV3(num_classes=NUM_CLASSES, use_aux=True).to(DEVICE)
print(model)

In [ ]:
# Cell 4 — Forward Pass Test (batch=2, 299×299)
dummy_input = torch.randn(2, 3, 299, 299).to(DEVICE)
model.eval()
output      = model(dummy_input)
print(f'Input  shape : {dummy_input.shape}')
print(f'Output shape : {output.shape}')

In [ ]:
# Cell 5 — Count Parameters
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
non_trainable    = total_params - trainable_params

print(f"{'='*40}")
print(f"  Total parameters      : {total_params:,}")
print(f"  Trainable parameters  : {trainable_params:,}")
print(f"  Non-trainable params  : {non_trainable:,}")
print(f"{'='*40}")

In [ ]:
# Cell 6 — Layer-wise Parameter Breakdown
print(f"{'Layer':<40} {'Shape':<30} {'Params':>10}")
print('-' * 82)
for name, param in model.named_parameters():
    print(f"{name:<40} {str(list(param.shape)):<30} {param.numel():>10,}")
print('-' * 82)
print(f"{'TOTAL':<71} {total_params:>10,}")

---
## Training

In [ ]:
# Cell 7 — Dataset & DataLoader
DATA_DIR   = './data'
BATCH_SIZE = 32

train_transform = transforms.Compose([
    transforms.Resize((299, 299)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
val_transform = transforms.Compose([
    transforms.Resize((299, 299)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

train_dataset = datasets.ImageFolder(f'{DATA_DIR}/train', transform=train_transform)
val_dataset   = datasets.ImageFolder(f'{DATA_DIR}/val',   transform=val_transform)
train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=4)
val_loader    = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

CLASS_NAMES = train_dataset.classes
print(f'Classes    : {CLASS_NAMES}')
print(f'Train size : {len(train_dataset)}')
print(f'Val size   : {len(val_dataset)}')

In [ ]:
# Cell 8 — Training Loop
EPOCHS = 20
LR     = 0.001

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)

history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss, correct, total = 0.0, 0, 0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for images, labels in loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            if train: optimizer.zero_grad()
            out = model(images)
            if train and isinstance(out, tuple):
                main_out, aux_out = out
                loss = criterion(main_out, labels) + 0.3 * criterion(aux_out, labels)
            else:
                main_out = out if not isinstance(out, tuple) else out[0]
                loss = criterion(main_out, labels)
            if train: loss.backward(); optimizer.step()
            total_loss += loss.item() * images.size(0)
            correct    += main_out.max(1)[1].eq(labels).sum().item()
            total      += labels.size(0)
    return total_loss / total, 100.0 * correct / total

print(f"{'Epoch':<8} {'Tr Loss':<10} {'Tr Acc':<10} {'Val Loss':<10} {'Val Acc':<10}")
print('-' * 50)
for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc = run_epoch(train_loader, train=True)
    vl_loss, vl_acc = run_epoch(val_loader,   train=False)
    scheduler.step()
    history['train_loss'].append(tr_loss); history['val_loss'].append(vl_loss)
    history['train_acc'].append(tr_acc);  history['val_acc'].append(vl_acc)
    print(f'{epoch:<8} {tr_loss:<10.4f} {tr_acc:<10.2f} {vl_loss:<10.4f} {vl_acc:<10.2f}')

torch.save(model.state_dict(), 'inceptionv3_best.pth')
print('\nModel saved: inceptionv3_best.pth')

---
## Training Curves

In [ ]:
# Cell 9 — Training Curves (Loss & Accuracy)
epochs_range = range(1, EPOCHS + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Inception V3 — Training Curves', fontsize=14, fontweight='bold')

axes[0].plot(epochs_range, history['train_loss'], 'b-o', linewidth=2, markersize=4, label='Train Loss')
axes[0].plot(epochs_range, history['val_loss'],   'r-o', linewidth=2, markersize=4, label='Val Loss')
axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_range, history['train_acc'], 'b-o', linewidth=2, markersize=4, label='Train Acc')
axes[1].plot(epochs_range, history['val_acc'],   'r-o', linewidth=2, markersize=4, label='Val Acc')
axes[1].set_title('Accuracy'); axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy (%)')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: training_curves.png')

---
## Inference

In [ ]:
# Cell 10 — Inference on a Single Image
from PIL import Image

def predict_image(image_path, top_k=5):
    model.eval()
    transform = transforms.Compose([
        transforms.Resize((299, 299)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])
    img    = Image.open(image_path).convert('RGB')
    tensor = transform(img).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        probs = F.softmax(model(tensor), dim=1)

    top_probs, top_idx = torch.topk(probs, top_k, dim=1)
    top_probs = top_probs[0].cpu().tolist()
    top_idx   = top_idx[0].cpu().tolist()

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].imshow(img); axes[0].axis('off'); axes[0].set_title('Input Image')
    labels = [CLASS_NAMES[i] for i in top_idx]
    colors = ['#F44336' if i == 0 else '#2196F3' for i in range(top_k)]
    bars   = axes[1].barh(labels[::-1], [p*100 for p in top_probs[::-1]], color=colors[::-1])
    axes[1].set_xlabel('Confidence (%)'); axes[1].set_title(f'Top-{top_k} Predictions')
    for bar, prob in zip(bars, top_probs[::-1]):
        axes[1].text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
                     f'{prob*100:.1f}%', va='center')
    plt.tight_layout()
    plt.savefig('inference_result.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'\nPredicted: {CLASS_NAMES[top_idx[0]]}  ({top_probs[0]*100:.2f}%)')

# predict_image('your_image.jpg')
print('Call: predict_image("your_image.jpg")')

---
## ROC AUC Curve

In [ ]:
# Cell 11 — Collect Validation Predictions
model.eval()
all_probs  = []
all_labels = []

with torch.no_grad():
    for images, labels in val_loader:
        probs = F.softmax(model(images.to(DEVICE)), dim=1)
        all_probs.append(probs.cpu().numpy())
        all_labels.append(labels.numpy())

all_probs  = np.concatenate(all_probs,  axis=0)
all_labels = np.concatenate(all_labels, axis=0)
print(f'Predictions shape : {all_probs.shape}')
print(f'Labels shape      : {all_labels.shape}')

In [ ]:
# Cell 12 — ROC AUC Curve (One-vs-Rest, per class)
y_bin   = label_binarize(all_labels, classes=list(range(NUM_CLASSES)))
fig, ax = plt.subplots(figsize=(10, 8))
colors  = plt.cm.tab10(np.linspace(0, 1, NUM_CLASSES))

roc_auc_scores = {}
for i in range(NUM_CLASSES):
    fpr, tpr, _ = roc_curve(y_bin[:, i], all_probs[:, i])
    roc_auc     = auc(fpr, tpr)
    roc_auc_scores[CLASS_NAMES[i]] = roc_auc
    ax.plot(fpr, tpr, color=colors[i], linewidth=2,
            label=f'{CLASS_NAMES[i]}  (AUC = {roc_auc:.3f})')

macro_auc = roc_auc_score(y_bin, all_probs, average='macro', multi_class='ovr')
ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random (AUC = 0.500)')
ax.set_xlim([0.0, 1.0]); ax.set_ylim([0.0, 1.05])
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title(f'Inception V3 — ROC AUC Curve (Macro AUC = {macro_auc:.3f})', fontsize=13, fontweight='bold')
ax.legend(loc='lower right', fontsize=9); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('roc_auc_curve.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nMacro AUC : {macro_auc:.4f}')
print('\nPer-class AUC:')
for cls, score in roc_auc_scores.items():
    print(f'  {cls:<20} {score:.4f}')